# 用模型辅助检查回答

LLM-as-Judge 是让另一个模型根据明确规则检查回答。它适合处理“是否回应问题”、“是否有资料支持”这类很难用一个关键词判断的项目，但模型评分也会错。

本页分开呈现两部分：前面的 Cohen’s Kappa 数组是手写标签，只演示一致性计算；后面的两条校准样本会使用根目录 `.env` 中的 `glm-4-flash` 做真实 LLM-as-Judge 调用。校准样本一条使用教程参考答案作为有据样本，另一条故意加入原文没有的结论；它们只用于检查评审器，不代表任何 RAG 方法已经改善。

## 先固定评分规则

一个可复查的规则至少包含：输入有哪些，每个分数表示什么，应该返回哪些字段，以及资料不足时怎样处理。可以要求模型先列出支持或反对的原文，再给分；但最终还要比较它和人工判断的一致程度。

In [1]:
human = [1, 1, 0, 1, 0, 0, 1, 0]
judge = [1, 1, 1, 1, 0, 0, 1, 0]

def cohen_kappa(left, right):
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    left_pos = sum(left) / len(left)
    right_pos = sum(right) / len(right)
    expected = left_pos * right_pos + (1-left_pos) * (1-right_pos)
    return (observed - expected) / (1 - expected)

print("一致数量：", sum(a == b for a, b in zip(human, judge)), "/", len(human))
print("Cohen's kappa：", round(cohen_kappa(human, judge), 3))
print("不一致的样本编号：", [index for index, (a, b) in enumerate(zip(human, judge), start=1) if a != b])

一致数量： 7 / 8
Cohen's kappa： 0.75
不一致的样本编号： [3]


先用人工检查的小批问题校准，再扩大使用。评分模型、Prompt 或分数定义变更后，一致性需要重新检查。本页的 8 个标签只用于演示计算，不是对某个真实评分模型的结论。



## LLM-as-Judge 的评分设计、直接评分和逐步评分

LLM-as-Judge（模型评审）适合检查“是否切题、是否有资料支持、是否覆盖多个要求”等难以用字符串规则判断的项目。它不是事实来源，也不是天然可靠的标注员。Prompt 至少要固定角色、输入字段、评分锚点、资料不足的处理、输出格式和拒答规则。

直接评分只要求模型返回分数，成本较低但难以复查；逐步评分可以先列出每项结论及其原文依据，再给分，更容易发现漏项，但输出更长、费用更高。比较两种 Prompt 时要使用同一批问题，再看平均分、分数波动和人工评分是否一致。一次逐步评分更高，不能说明它对所有问题都更准。还要防止模型偏爱排在前面的答案、较长的答案、自己生成的答案或特定格式；可以交换答案顺序、限制长度并抽样人工复核。

本页保留 8 个手写二分类标签演示 Cohen’s Kappa；它们不是外部评审模型的实测结果。真实模型调用只针对下一段的两条校准样本。


## LLM-as-Judge 的代码写法

下面先展示评审模型的输入、输出和格式检查代码写法；末尾的运行单元会固定同一问题、资料和两种回答，真实调用后保留原始返回并与人工预期逐条复核。

```python
import json

DIRECT_JUDGE_PROMPT = """你是评审员，不是回答者。根据问题、资料和回答评分。
0=没有回应问题或主要内容错误；1=部分回应或部分有资料支持；2=完整回应且主要结论都有资料依据。
只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"原文短语\"]}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

STEPWISE_JUDGE_PROMPT = """先把回答拆成不超过 5 条可核对的结论；为每条结论列出资料中的原文短语，找不到就写 NOTHING_FOUND；
按有据且切题的结论比例给 0/1/2 分。只输出 JSON，字段为 claims、score、reason。
问题：{question}\n资料：{context}\n回答：{answer}"""

def parse_judge_json(raw):
    text = str(raw).strip()
    fence = chr(96) * 3
    if text.startswith(fence + "json") and text.endswith(fence):
        text = text[len(fence + "json"):-len(fence)].strip()
    obj = json.loads(text)
    if not isinstance(obj, dict):
        raise ValueError("judge 输出必须是 JSON 对象")
    score = obj.get("score")
    if isinstance(score, bool) or score not in (0, 1, 2):
        raise ValueError("score 必须是 0、1 或 2")
    reason = obj.get("reason")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("reason 必须是非空字符串")
    claims = obj.get("claims", [])
    evidence = obj.get("evidence", [])
    if not isinstance(claims, list):
        raise ValueError("claims 必须是列表")
    if not isinstance(evidence, list):
        raise ValueError("evidence 必须是列表")
    if any(not isinstance(item, str) for item in evidence):
        raise ValueError("evidence 每项必须是字符串")
    if any(not item.strip() for item in evidence):
        raise ValueError("evidence 每项必须是非空字符串")
    for claim in claims:
        if not isinstance(claim, dict) or not isinstance(claim.get("evidence", []), list):
            raise ValueError("claims 中的 evidence 必须是列表")
    return {"score": score, "reason": reason, "evidence": evidence, "claims": claims}

def judge_one(question, context, answer, llm_call, stepwise=True):
    template = STEPWISE_JUDGE_PROMPT if stepwise else DIRECT_JUDGE_PROMPT
    return parse_judge_json(llm_call(template.format(question=question, context=context, answer=answer)))

import json

def cohen_kappa_multiclass(left, right):
    """不依赖 sklearn 的 Cohen's Kappa；输入为等长离散标签。"""
    if len(left) != len(right) or not left: raise ValueError("两组标签必须等长且非空")
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    labels = set(left) | set(right)
    expected = sum((left.count(label) / len(left)) * (right.count(label) / len(right)) for label in labels)
    return (observed - expected) / (1 - expected) if expected < 1 else 1.0

# 先用人工标注的小批样本校准，再扩大到整套评估集；换模型、Prompt 或标签定义后要重校准。
```


## 评分规则、两种问法和已知局限

先把评分协议写下来，再调用评审模型：输入固定为问题（`question`）、资料（`context`）、回答（`answer`），必要时另列参考答案（`ground_truth`）；0 分表示没有回答或主要结论错误，1 分表示部分回答或只有部分结论有依据，2 分表示完整回答且主要结论都有依据。输出必须包含分数、简短理由和原文证据；资料没有答案时，应该检查模型是否明确拒答，不能因为回答写得流畅就给高分。

| 做法 | 提示词要求 | 优点 | 风险 |
| --- | --- | --- | --- |
| 直接评分 | 读完输入后直接返回分数 | 省 token、速度快 | 理由少，漏掉某个结论后不易追查 |
| 逐步评分 | 先列可核对的结论，再为每条结论找原文证据，再汇总分数 | 更容易发现漏答和无依据结论 | 输出长、费用高，模型的中间分析仍可能出错 |

两种做法必须在同一批问题、相同资料、相同分数定义上比较；一次实验看到逐步评分更高，不能推出它在所有模型和任务上都更好。还要防止位置偏差、冗长偏差、自我偏好和格式偏差：交换候选答案顺序、限制长度、抽样人工复核，并把评审模型当作有误差的测量工具。

In [2]:
import json

def cohen_kappa_multiclass(left, right):
    """只用标准库计算离散标签的一致性；两组标签必须来自同一批样本。"""
    if len(left) != len(right) or not left:
        raise ValueError("两组标签必须等长且非空")
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    labels = set(left) | set(right)
    expected = sum((left.count(label) / len(left)) * (right.count(label) / len(right))
                    for label in labels)
    return (observed - expected) / (1 - expected) if expected < 1 else 1.0

def summarize_judge_modes(direct_scores, stepwise_scores, human_scores=None):
    """汇总两种评分的均值、差异和（可选）与人工标签的 Kappa。"""
    if len(direct_scores) != len(stepwise_scores):
        raise ValueError("直接评分和逐步评分必须使用同一批样本")
    result = {
        "n": len(direct_scores),
        "direct_mean": sum(direct_scores) / len(direct_scores) if direct_scores else 0.0,
        "stepwise_mean": sum(stepwise_scores) / len(stepwise_scores) if stepwise_scores else 0.0,
        "stepwise_minus_direct": (sum(stepwise_scores) - sum(direct_scores)) / len(direct_scores)
            if direct_scores else 0.0,
    }
    if human_scores is not None:
        if len(human_scores) != len(direct_scores):
            raise ValueError("人工标签必须与同一批样本对齐")
        result["direct_kappa"] = cohen_kappa_multiclass(human_scores, direct_scores)
        result["stepwise_kappa"] = cohen_kappa_multiclass(human_scores, stepwise_scores)
    return result

direct_demo = [2, 1, 0, 2]
stepwise_demo = [2, 2, 0, 2]
human_demo = [2, 1, 0, 2]
summary = summarize_judge_modes(direct_demo, stepwise_demo, human_demo)
print(
    f"评分对照：{summary['n']} 个问题；直接评分平均 {summary['direct_mean']:.2f}，"
    f"逐步评分平均 {summary['stepwise_mean']:.2f}，逐步评分高 {summary['stepwise_minus_direct']:.2f}；"
    f"与人工标签的一致性分别为 {summary['direct_kappa']:.2f} 和 {summary['stepwise_kappa']:.2f}。"
)
# 上面的标签是本地演示数据，不调用评审模型，也不替换前面已保存的 8 个标签输出。

PROBE_DIRECT_PROMPT = "只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"原文短语\"]}}。问题：{question}\n资料：{context}\n回答：{answer}"
def judge_one(question, context, answer, llm_call, stepwise=True):
    # 双花括号让 format 保留 JSON 对象；此探针只验证模板，不计入真实校准。
    prompt = PROBE_DIRECT_PROMPT.format(question=question, context=context, answer=answer)
    return json.loads(llm_call(prompt))

probe_prompt = PROBE_DIRECT_PROMPT.format(question='一个有依据的问题', context='资料片段', answer='依据资料作答')
assert all(label in probe_prompt for label in ('问题：', '资料：', '回答：'))
print('格式探针：直接评分模板可完成字段插值；不调用评审模型，也不计入下面的真实校准。')


评分对照：4 个问题；直接评分平均 1.25，逐步评分平均 1.50，逐步评分高 0.25；与人工标签的一致性分别为 1.00 和 0.56。
格式探针：直接评分模板可完成字段插值；不调用评审模型，也不计入下面的真实校准。


## 第一次真实校准的失败记录（保留）

上一轮使用同一问题、同一页资料和两条回答完成了 4 次 `glm-4-flash` 调用。下面保留它的失败过程，后面的改进复查不会把它改写成成功：有原文依据的回答，直接评分为 2、逐项评分为 2，均与人工预期一致；明确错误的回答，直接评分为 0，但逐项评分错误地给了 2，且把被资料语义反驳的结论列为有据。原逐项评分只有 1/2 条与人工预期一致。

这说明“出现相关词”不等于“支持该结论”：原逐项结果引用了包含相关词的句子，却没有判断回答的语义方向。改进后的复查只增加 2 次逐项评分调用，并单独做引用核验。

In [3]:
import json
import sys
from pathlib import Path

course_root = Path.cwd()
for folder in (course_root, *course_root.parents):
    if (folder / 'data' / 'dataset/manifest.json').is_file():
        course_root = folder
        break
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import load_query_catalog, load_pdf_pages
from common.nontraining_utils import llm_call, load_annotation

DIRECT_CALIBRATION_PROMPT = """你是严格的资料依据评审员，不是回答者。
0=没有回应问题或主要结论错误；1=部分回应或部分有资料支持；2=完整回应且主要结论都有资料依据。
只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"资料中的短语\"]}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

STEPWISE_CALIBRATION_PROMPT = """你是严格的资料依据评审员。先列出回答中不超过 5 条可核对的结论，再逐条从资料找证据；找不到就写 NOTHING_FOUND。最后按有据且切题的结论比例给 0/1/2 分；即使有三条结论，也只能给最高分 2，绝对不要输出 3。score 只能是整数 0、1 或 2。
只输出 JSON：{{\"claims\": [{{\"claim\": \"结论\", \"evidence\": [\"短语或 NOTHING_FOUND\"], \"supported\": true|false}}], \"score\": 0|1|2, \"reason\": \"一句话\"}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

def parse_real_judge(raw):
    text = str(raw).strip()
    if text.startswith('```json') and text.endswith('```'):
        text = text[len('```json'):-3].strip()
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError('模型返回的不是 JSON 对象')
    score = value.get('score')
    if isinstance(score, bool) or score not in (0, 1, 2):
        raise ValueError('模型返回的分数不是 0、1、2')
    reason = value.get('reason')
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError('模型返回的 reason 必须是非空字符串')
    if 'evidence' in value and not isinstance(value['evidence'], list):
        raise ValueError('模型返回的 evidence 必须是列表')
    if 'evidence' in value and any(not isinstance(item, str) for item in value['evidence']):
        raise ValueError('模型返回的 evidence 每项必须是字符串')
    if 'claims' in value and not isinstance(value['claims'], list):
        raise ValueError('模型返回的 claims 必须是列表')
    return value

call_attempts = []
def call_model(prompt):
    call_attempts.append('ZHIPUAI_API_KEY')
    raw = llm_call(prompt, max_tokens=700)
    return raw, 'ZHIPUAI_API_KEY'

def run_real_judge(sample, stepwise):
    template = STEPWISE_CALIBRATION_PROMPT if stepwise else DIRECT_CALIBRATION_PROMPT
    prompt = template.format(question=sample['question'], context=sample['context'], answer=sample['answer'])
    raw, key_source = call_model(prompt)
    parsed = parse_real_judge(raw)
    return {'status': '成功', 'raw': raw, 'key_source': key_source,
            'parsed': parsed, 'score': parsed['score']}

case = next(item for item in load_query_catalog() if item['id'] == 'contextual_cv_three_methods')
page18 = next(item['text'] for item in load_pdf_pages() if item['page'] == 18)
case_annotation = load_annotation(case['id'])
samples = [
    {
        'label': '有原文依据的校准样本',
        'question': case['query'],
        'context': page18,
        'answer': case_annotation['reference_answer'],
        'human_score': 2,
        'human_expectation': '完整且有原文依据',
    },
    {
        'label': '含明确错误结论的校准样本',
        'question': case['query'],
        'context': page18,
        'answer': '第 2.2 节只介绍留出法；交叉验证法和自助法属于别的章节，所以模型评估不需要它们。',
        'human_score': 0,
        'human_expectation': '主要结论错误，资料没有支持',
    },
]

real_results = []
print('真实校准模型：glm-4-flash；同一问题、同一页资料、两条回答；目标调用 4 次。')
for sample in samples:
    print('\n样本类型：', sample['label'])
    print('问题：', sample['question'])
    print('回答：', sample['answer'])
    print('人工预期：', sample['human_expectation'], '；人工分数：', sample['human_score'])
    for mode_name, stepwise in (('直接评分', False), ('逐项核对评分', True)):
        result = run_real_judge(sample, stepwise)
        real_results.append((sample, mode_name, result))
        print(mode_name, '状态：', result['status'], '；调用来源：', result['key_source'])
        parsed = result['parsed']
        print('模型分数：', result['score'], '；简短理由：', parsed['reason'])
        print('模型返回证据：', json.dumps(parsed.get('evidence', []), ensure_ascii=False))
        for index, item in enumerate(parsed.get('claims', []), start=1):
            if not isinstance(item, dict):
                raise ValueError('claims 中每项必须是对象')
            claim_text = item.get('claim', '')
            claim_evidence = json.dumps(item.get('evidence', []), ensure_ascii=False)
            claim_support = item.get('supported', '未说明')
            print(f'第 {index} 条结论：{claim_text}；证据：{claim_evidence}；资料支持：{claim_support}')
        print('与人工预期一致：', result['score'] == sample['human_score'])

success_count = sum(result['status'] == '成功' for _, _, result in real_results)
print('调用模型：glm-4-flash；实际调用尝试次数：', len(call_attempts), '；成功返回次数：', success_count)
if len(real_results) != 4 or success_count != 4:
    raise AssertionError('四次真实校准调用必须全部成功并保存结构化结果')
print('校准结论：两条人工预期和四次评分结果均已保存，可逐条复核；这不是 RAG 方法效果比较。')

真实校准模型：glm-4-flash；同一问题、同一页资料、两条回答；目标调用 4 次。

样本类型： 有原文依据的校准样本
问题： 第2.2节列出的三种模型评估办法分别叫什么？
回答： 第 2.2 节介绍留出法、交叉验证法和自助法。
人工预期： 完整且有原文依据 ；人工分数： 2


直接评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 完整回应且主要结论都有资料依据
模型返回证据： ["留出法", "交叉验证法", "自助法"]
与人工预期一致： True


逐项核对评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 文中明确提到了三种模型评估方法：留出法、交叉验证法和自助法。
模型返回证据： []
第 1 条结论：第 2.2 节介绍留出法；证据：["留出法"]；资料支持：True
第 2 条结论：第 2.2 节介绍交叉验证法；证据：["交叉验证法"]；资料支持：True
第 3 条结论：第 2.2 节介绍自助法；证据：["自助法"]；资料支持：True
与人工预期一致： True

样本类型： 含明确错误结论的校准样本
问题： 第2.2节列出的三种模型评估办法分别叫什么？
回答： 第 2.2 节只介绍留出法；交叉验证法和自助法属于别的章节，所以模型评估不需要它们。
人工预期： 主要结论错误，资料没有支持 ；人工分数： 0


直接评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 0 ；简短理由： 主要结论错误
模型返回证据： ["资料中明确提到第2.2节介绍了三种模型评估方法：留出法、交叉验证法、自助法。"]
与人工预期一致： True


逐项核对评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 有两条结论有证据支持，但其中一条结论的关联性较弱。
模型返回证据： []
第 1 条结论：第 2.2 节只介绍留出法；证据：["留出法", "留出法由于操作简单，因此最常用"]；资料支持：True
第 2 条结论：交叉验证法属于别的章节；证据：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；资料支持：True
第 3 条结论：自助法属于别的章节；证据：["自助法常用于集成学习（详见“西瓜书”第 8 章的 8.2 节和 8.3 节）产生基分类器"]；资料支持：True
第 4 条结论：模型评估不需要交叉验证法和自助法；证据：["留出法和自助法简单易懂，在此不再赘述", "自助法常用于集成学习"]；资料支持：False
第 5 条结论：模型评估不需要交叉验证法；证据：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；资料支持：False
与人工预期一致： False
调用模型：glm-4-flash；实际调用尝试次数： 4 ；成功返回次数： 4
校准结论：两条人工预期和四次评分结果均已保存，可逐条复核；这不是 RAG 方法效果比较。


## 发现误判后怎样修正：语义关系和引用核验

改进的逐项评分要求每条结论标为“资料支持（supported）”“资料反驳（contradicted）”或“资料未提及（not_found）”。引用必须是资料中的原文短语；出现相同词语不代表支持，评审器还要判断资料和回答的语义方向。只要主要结论被资料直接反驳，总分就不能是 2。

代码会逐条检查引用短语（忽略空白后）是否真的出现在资料中；解析错误、非法关系或找不到的短语都会直接抛错，不会静默算作支持。改进复查仍只把问题、资料和回答传给模型，人工分数只在模型返回后用于对照。

In [4]:
import re

IMPROVED_STEPWISE_PROMPT = """你是严格的资料依据评审员，不是回答者。只根据问题、资料和回答判断，不要使用参考答案或人工分数。
先把回答拆成不超过 5 条主要结论。每条结论必须填写 relation：supported（资料支持）、contradicted（资料反驳）或 not_found（资料未提及）三者之一；证据只能逐字摘自资料中的短语，不得用你的改写。出现相关词语不等于支持，必须判断资料和回答的语义方向。若资料直接反驳主要结论，relation 必须为 contradicted，且总分绝对不能是 2。relation=not_found 时 evidence 必须严格是 ["NOTHING_FOUND"]，不能写解释性句子。
总分只能是整数 0、1 或 2：0=主要结论错误或被反驳，1=部分正确或部分有据，2=完整且主要结论均有据。只输出 JSON：{{\"claims\": [{{\"claim\": \"结论\", \"relation\": \"supported|contradicted|not_found\", \"evidence\": [\"原文短语或 NOTHING_FOUND\"]}}], \"score\": 0|1|2, \"reason\": \"一句话\"}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

RELATION_LABELS = {'supported': '资料支持', 'contradicted': '资料反驳', 'not_found': '资料未提及'}

def parse_improved_judge(raw):
    value = parse_real_judge(raw)
    claims = value.get('claims')
    if not isinstance(claims, list) or not claims:
        raise ValueError('逐项结果缺少 claims 列表')
    for item in claims:
        if not isinstance(item, dict) or set(item) != {'claim', 'relation', 'evidence'}:
            raise ValueError('逐项结果每条 claim 必须严格包含 claim/relation/evidence')
        if not isinstance(item['claim'], str) or not item['claim'].strip():
            raise ValueError('逐项结果的 claim 必须是非空字符串')
        if item['relation'] not in RELATION_LABELS:
            raise ValueError('逐项结果的 relation 不在允许范围内')
        if not isinstance(item['evidence'], list):
            raise ValueError('逐项结果的 evidence 必须是列表')
        if any(not isinstance(phrase, str) for phrase in item['evidence']):
            raise ValueError('逐项结果的 evidence 每项必须是字符串')
        if any(not phrase.strip() for phrase in item['evidence']):
            raise ValueError('逐项结果的 evidence 每项必须是非空字符串')
        if item['relation'] == 'not_found' and item['evidence'] != ['NOTHING_FOUND']:
            raise ValueError('not_found 的 evidence 必须严格为 [NOTHING_FOUND]')
        if item['relation'] in {'supported', 'contradicted'} and not item['evidence']:
            raise ValueError('supported/contradicted 必须提供原文 evidence')
    return value

def normalize_for_evidence(value):
    return re.sub(r'\s+', '', str(value or ''))

def verify_evidence(parsed, context):
    normalized_context = normalize_for_evidence(context)
    rows = []
    for index, item in enumerate(parsed['claims'], start=1):
        relation = item['relation']
        phrases = [normalize_for_evidence(phrase) for phrase in item['evidence']]
        if relation == 'not_found':
            if item['evidence'] != ['NOTHING_FOUND']:
                raise ValueError(f'第 {index} 条 not_found evidence 必须严格为 NOTHING_FOUND')
            found = []
            missing = []
            valid = True
            status = '未引用（模型声明资料未提及）'
        else:
            found = [phrase for phrase in phrases if phrase in normalized_context]
            missing = [phrase for phrase in phrases if phrase not in normalized_context]
            if not phrases or missing:
                raise ValueError(f'第 {index} 条 {relation} 的 evidence 未逐字命中资料：{missing!r}')
            valid = True
            status = '引用全部在资料中找到'
        rows.append({'index': index, 'claim': item.get('claim', ''), 'relation': relation,
                     'evidence': item['evidence'], 'found': found, 'missing': missing,
                     'valid': valid, 'status': status})
    if any(row['relation'] == 'contradicted' for row in rows) and parsed['score'] == 2:
        raise ValueError('存在 contradicted 主要结论时 score 不能为 2')
    return rows, True

def run_improved_stepwise(sample):
    prompt = IMPROVED_STEPWISE_PROMPT.format(question=sample['question'], context=sample['context'], answer=sample['answer'])
    raw, key_source = call_model(prompt)
    parsed = parse_improved_judge(raw)
    rows, contradiction_guard = verify_evidence(parsed, sample['context'])
    return {'status': '成功', 'raw': raw, 'key_source': key_source,
            'parsed': parsed, 'score': parsed['score'], 'rows': rows,
            'contradiction_guard': contradiction_guard,
            'evidence_verifiable': all(row['valid'] for row in rows)}

improved_start_attempts = len(call_attempts)
improved_results = []
print('改进逐项复查：只增加同一两条样本的 2 次 glm-4-flash 调用；模型输入只有问题、资料和回答。')
for sample in samples:
    result = run_improved_stepwise(sample)
    improved_results.append((sample, result))
    print('\n样本类型：', sample['label'], '；调用来源：', result.get('key_source') or '无', '；状态：', result['status'])
    parsed = result['parsed']
    print('模型分数：', result['score'], '；简短理由：', parsed['reason'])
    for row in result['rows']:
        missing_text = json.dumps(row['missing'], ensure_ascii=False) if row['missing'] else '无'
        print(f"第 {row['index']} 条结论：{row['claim']}；关系：{RELATION_LABELS[row['relation']]}；引用：{json.dumps(row['evidence'], ensure_ascii=False)}；代码核验：{row['status']}；未命中引用：{missing_text}")
    print('引用全部可核验：', result['evidence_verifiable'], '；证据与反驳约束均通过：', result['contradiction_guard'])
    print('与人工预期一致：', result['score'] == sample['human_score'])

improved_correct = sum(result['status'] == '成功' and result['score'] == sample['human_score']
                       for sample, result in improved_results)
improved_evidence_verifiable = sum(result['status'] == '成功' and result['evidence_verifiable']
                                   for _, result in improved_results)
improved_calls = len(call_attempts) - improved_start_attempts
if improved_calls != 2 or improved_correct != 2 or improved_evidence_verifiable != 2:
    raise AssertionError('两次改进复查必须成功解析、引用全部可核验且分数符合固定校准预期')
print('\n比较：原始逐项评分的最终分数正确 1/2；改进后最终分数正确', f'{improved_correct}/2，',
      '两条结果的引用全部可核验。')
print('额外调用：', improved_calls, '次 glm-4-flash；相对原始记录新增这些请求，未读取计费价格。')
print('限制：只有两条校准样本，不能证明改进后的 Prompt 对一般任务普遍更好。')

改进逐项复查：只增加同一两条样本的 2 次 glm-4-flash 调用；模型输入只有问题、资料和回答。



样本类型： 有原文依据的校准样本 ；调用来源： ZHIPUAI_API_KEY ；状态： 成功
模型分数： 2 ；简短理由： 所有主要结论均有据
第 1 条结论：第 2.2 节介绍留出法；关系：资料支持；引用：["留出法由于操作简单，因此最常用"]；代码核验：引用全部在资料中找到；未命中引用：无
第 2 条结论：第 2.2 节介绍交叉验证法；关系：资料支持；引用：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；代码核验：引用全部在资料中找到；未命中引用：无
第 3 条结论：第 2.2 节介绍自助法；关系：资料支持；引用：["自助法常用于集成学习（详见“西瓜书”第 8 章的 8.2 节和 8.3 节）产生基分类器"]；代码核验：引用全部在资料中找到；未命中引用：无
引用全部可核验： True ；证据与反驳约束均通过： True
与人工预期一致： True



样本类型： 含明确错误结论的校准样本 ；调用来源： ZHIPUAI_API_KEY ；状态： 成功
模型分数： 0 ；简短理由： 回答中的结论与资料中的信息不符，资料明确指出第 2.2 节介绍了三种模型评估方法，而回答中提到不需要这些方法，资料中并未提及这一点。
第 1 条结论：第 2.2 节只介绍留出法；关系：资料反驳；引用：["本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法"]；代码核验：引用全部在资料中找到；未命中引用：无
第 2 条结论：交叉验证法和自助法属于别的章节；关系：资料未提及；引用：["NOTHING_FOUND"]；代码核验：未引用（模型声明资料未提及）；未命中引用：无
第 3 条结论：所以模型评估不需要它们；关系：资料未提及；引用：["NOTHING_FOUND"]；代码核验：未引用（模型声明资料未提及）；未命中引用：无
引用全部可核验： True ；证据与反驳约束均通过： True
与人工预期一致： True

比较：原始逐项评分的最终分数正确 1/2；改进后最终分数正确 2/2， 两条结果的引用全部可核验。
额外调用： 2 次 glm-4-flash；相对原始记录新增这些请求，未读取计费价格。
限制：只有两条校准样本，不能证明改进后的 Prompt 对一般任务普遍更好。


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[执行端到端验收](端到端验收.ipynb)

